# 10 JUNE 2026

In [ ]:
!pip3 install pandas numpy

In [ ]:
!pip3 install imbalanced-learn

In [ ]:
!pip3 install scikit-learn

In [ ]:
!pip3 install tensorflow

In [ ]:
!pip3 install streamlit

In [12]:
import pandas as pd
import numpy as np

from imblearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,roc_curve)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

In [7]:
# ******* Load Data *********
df = pd.read_csv(r"C:\Users\nisha\OneDrive\Desktop\Nisha\GUVI\FinalProject\BankTermDepositPrediction\Data\bank-full.csv", sep=';')

df.drop('duration', axis=1, inplace=True)

# ******* Target encoding *********
df['y'] = df['y'].map({'no':0,'yes':1})

# ******* FEATURE ENGINNERING *********
def create_features(df):
    df = df.copy()
    df['contacted_before'] = np.where(df['previous'] > 0,1,0)
    df['total_loans'] = (
                            (df['housing'] == 'yes').astype(int)+
                            (df['loan'] == 'yes').astype(int)
                        )
    df['age_balance'] = (df['age']*df['balance'])
    return df

# ******* FEATURES & TARGET VARIABLE SEPARATION *********
X = df.drop('y', axis=1)
y = df['y']

X = create_features(X)

In [8]:
# ******* FEATURES Details *********
numeric_features = ['age','balance','day','campaign','pdays','previous','age_balance','total_loans','contacted_before']
ordinal_features = ['education']
categorical_features = ['job','marital','default','housing','loan','contact','month','poutcome']

numeric_transformer = Pipeline([
                                    ('imputer', SimpleImputer(strategy='median')),
                                    ('scaler', RobustScaler())
                                ])

ordinal_transformer = Pipeline([
                                    ('imputer', SimpleImputer(strategy='most_frequent')),
                                    ('ordinal', OrdinalEncoder(categories=[['unknown','primary','secondary','tertiary']]))
                                ])

categorical_transformer = Pipeline([
                                        ('imputer', SimpleImputer(strategy='most_frequent')),
                                        ('onehot', OneHotEncoder(handle_unknown='ignore'))
                                    ])

preprocessor = ColumnTransformer([
                                    ('num',numeric_transformer,numeric_features),
                                    ('ord',ordinal_transformer,ordinal_features),
                                    ('cat',categorical_transformer,categorical_features)
                                ])

In [9]:
# ******* Train-test Split *********
X_train,X_test,y_train,y_test = train_test_split(X, y,test_size=0.2,random_state=42,stratify=y)

In [10]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [11]:
classes = np.unique(y_train)

class_weights = compute_class_weight(
                                        class_weight='balanced',
                                        classes=classes,
                                        y=y_train
                                    )

class_weight_dict = {
                        0: class_weights[0],
                        1: class_weights[1]
                    }

print(class_weight_dict)

{0: np.float64(0.5662397845758838), 1: np.float64(4.27416686362562)}


# Aritifical Neural Network (ANN) - Single layer

In [13]:
model = Sequential([
                        Dense(32, activation='relu', input_shape=(X_train_processed.shape[1],)),
                        Dense(1, activation='sigmoid')
                    ])

model.compile(
                optimizer='adam',
                loss='binary_crossentropy',
                metrics=[
                    tf.keras.metrics.AUC(name='auc')
                ]
            )

model.summary()

c:\Users\nisha\OneDrive\Desktop\Nisha\GUVI\FinalProject\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │         1,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,665 (6.50 KB)

 Trainable params: 1,665 (6.50 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
early_stop = EarlyStopping(
                            monitor='val_auc',
                            mode='max',
                            patience=10,
                            restore_best_weights=True
                        )

history = model.fit(
                        X_train_processed,
                        y_train,
                        validation_split=0.2,
                        epochs=100,
                        batch_size=256,
                        class_weight=class_weight_dict,
                        callbacks=[early_stop],
                        verbose=1
                    )
y_prob = model.predict(X_test_processed).flatten()

threshold = 0.35
y_pred = (y_prob >= threshold).astype(int)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

Epoch 1/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - auc: 0.6506 - loss: 0.7656 - val_auc: 0.7024 - val_loss: 0.6435
Epoch 2/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - auc: 0.7325 - loss: 0.6214 - val_auc: 0.7399 - val_loss: 0.6059
Epoch 3/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - auc: 0.7590 - loss: 0.5861 - val_auc: 0.7558 - val_loss: 0.5723
Epoch 4/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - auc: 0.7680 - loss: 0.5749 - val_auc: 0.7572 - val_loss: 0.5851
Epoch 5/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - auc: 0.7725 - loss: 0.5694 - val_auc: 0.7707 - val_loss: 0.5528
Epoch 6/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - auc: 0.7757 - loss: 0.5647 - val_auc: 0.7606 - val_loss: 0.5773
Epoch 7/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - auc: 0.7757 - loss: 0.5643 - val_auc: 0.7712 - val_loss: 0.5034
Epoch 8/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - auc: 0.7789 - loss: 0.5598 - val_auc: 0.7735 - val_loss: 0.5397
Epoch 9/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/

# Deep Neural Network (DNN) - Multiple layer

In [15]:
dnn_model = Sequential([
                        Dense(128, activation='relu',input_shape=(X_train_processed.shape[1],)),
                        BatchNormalization(),
                        Dropout(0.3),

                        Dense(64, activation='relu'),
                        BatchNormalization(),
                        Dropout(0.3),

                        Dense(32, activation='relu'),

                        Dense(1, activation='sigmoid')
                    ])

dnn_model.compile(
                optimizer='adam',
                loss='binary_crossentropy',
                metrics=[
                    tf.keras.metrics.AUC(name='auc')
                ]
            )

dnn_model.summary()

c:\Users\nisha\OneDrive\Desktop\Nisha\GUVI\FinalProject\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                 │ (None, 128)            │         6,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,665 (69.00 KB)

 Trainable params: 17,281 (67.50 KB)

 Non-trainable params: 384 (1.50 KB)

In [16]:
early_stop = EarlyStopping(
                            monitor='val_auc',
                            mode='max',
                            patience=10,
                            restore_best_weights=True
                        )

history = dnn_model.fit(
                        X_train_processed,
                        y_train,
                        validation_split=0.2,
                        epochs=100,
                        batch_size=256,
                        class_weight=class_weight_dict,
                        callbacks=[early_stop],
                        verbose=1
                    )
y_prob = dnn_model.predict(X_test_processed).flatten()

threshold = 0.35
y_pred = (y_prob >= threshold).astype(int)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

Epoch 1/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - auc: 0.6521 - loss: 0.6840 - val_auc: 0.7015 - val_loss: 0.5578
Epoch 2/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - auc: 0.7032 - loss: 0.6349 - val_auc: 0.7306 - val_loss: 0.5338
Epoch 3/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - auc: 0.7330 - loss: 0.6087 - val_auc: 0.7480 - val_loss: 0.4941
Epoch 4/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - auc: 0.7466 - loss: 0.5953 - val_auc: 0.7607 - val_loss: 0.5016
Epoch 5/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - auc: 0.7578 - loss: 0.5826 - val_auc: 0.7630 - val_loss: 0.5362
Epoch 6/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - auc: 0.7601 - loss: 0.5827 - val_auc: 0.7771 - val_loss: 0.5221
Epoch 7/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - auc: 0.7651 - loss: 0.5772 - val_auc: 0.7698 - val_loss: 0.5717
Epoch 8/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - auc: 0.7622 - loss: 0.5790 - val_auc: 0.7735 - val_loss: 0.5562
Epoch 9/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms